# TBP Family Office Data Processing

Goal: build one master file from the 3 `TBP_*.xlsx` sources.

Step 1: load the `Master Longlist` sheet from the Fresh Master file as the base schema, then compare its columns against the other two files' longlist sheets to find mismatches.

In [1]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("..")  # TBP root, one level up from tbp-dashboard/

FILES = {
    "master": DATA_DIR / "TBP_Family_Office_Generated_Lists_Fresh_Master.xlsx",
    "sea": DATA_DIR / "TBP_SEA_Medium sized.xlsx",
    "uzkz": DATA_DIR / "TBP_UZ-KZ_Independent_Corridor_Investor_Longlist.xlsx",
}

for key, path in FILES.items():
    print(key, "->", path.resolve(), "exists:", path.exists())

master -> C:\Users\USER\Desktop\TBP\TBP_Family_Office_Generated_Lists_Fresh_Master.xlsx exists: True
sea -> C:\Users\USER\Desktop\TBP\TBP_SEA_Medium sized.xlsx exists: True
uzkz -> C:\Users\USER\Desktop\TBP\TBP_UZ-KZ_Independent_Corridor_Investor_Longlist.xlsx exists: True


## Load the base (Master Longlist)

In [2]:
master_df = pd.read_excel(FILES["master"], sheet_name="Master Longlist")

print(master_df.shape)
master_df.head()

(113, 30)


,Region,Country,Organisation,Prospect Category,HQ / Primary Geography,Address / Office Location,Family / Founder / Strategic Nature,Known Sector Themes,TBP / Regional Corridor Relevance,Possible TBP Entry Point,...,Priority,Pipeline Stage,Scoring Status,Public Source URLs,Notes / Diligence Flags,Email address,Contact Email Status,Contact Email Source URLs,Contact Email Notes,Contact Enrichment Date
0,Central Asia,Kazakhstan,Verny Capital / Bulat Utemuratov,Private investment group / family-capital styl...,Almaty / Astana,NaN,Private investment group / family-capital styl...,Family office services / private wealth / stew...,Leader in private investments in Kazakhstan; i...,Central Asia / Middle Corridor; Uzbekistan-Kaz...,...,Priority,Identified,Indicative scored – review required,https://vernycapital.com/en/bulat-utemuratov-i...,Apply enhanced due diligence where political e...,info@vernycapital.com,Official/public email - verify before outreach,https://vernycapital.com/en/bulat-utemuratov-i...,Use general office email or warm introduction;...,2026-07-01
1,Central Asia,Kazakhstan,BI Group / Aidyn Rakhimbayev,"Investment, development, construction and educ...",Astana,NaN,"Investment, development, construction and educ...",Family office services / private wealth / stew...,"City development, real estate, construction an...",Central Asia / Middle Corridor; Uzbekistan-Kaz...,...,Priority,Identified,Indicative scored – review required,https://bi.group/en/company,Apply enhanced due diligence where political e...,ipm@bi.group,Official/public email - verify before outreach,https://bi.group/en/company,Use institutional or partnership route; verify...,2026-07-01
2,Central Asia,Kazakhstan,Lancaster Group,Diversified international holding company,Almaty,NaN,Diversified international holding company,"Mining, infrastructure, energy, financial serv...","Industrial infrastructure, oil and gas service...",Central Asia / Middle Corridor; Uzbekistan-Kaz...,...,Priority,Identified,Indicative scored – review required,https://lancasterholding.com/en,Apply enhanced due diligence where political e...,info@lgk.kz,Official/public email - verify before outreach,https://lancasterholding.com/en,Use general office email; verify current conta...,2026-07-01
3,Central Asia,Kazakhstan,Resmi Group,Diversified investment holding,Kazakhstan / Central Asia,NaN,Diversified investment holding,Family office services / private wealth / stew...,Diversified investment holding operating in Ka...,Central Asia / Middle Corridor; Uzbekistan-Kaz...,...,High,Identified,Indicative scored – review required,https://kz.linkedin.com/company/group-of-compa...,Apply enhanced due diligence where political e...,inform@resmi.kz,Official/public email - verify before outreach,https://kase.kz/en/listing/issuers/RESC/,Issuer/contact source; verify best route for c...,2026-07-01
4,Central Asia,Kazakhstan,AIFC Family Office Framework,Family office jurisdiction / structuring platform,Astana,NaN,Family office jurisdiction / structuring platform,Family office services / private wealth / stew...,Institutional route for structuring Central As...,Central Asia / Middle Corridor; Uzbekistan-Kaz...,...,Gateway,Identified,Indicative scored – review required,https://afsa.aifc.kz/aifc-introduces-family-of...,Apply enhanced due diligence where political e...,info@afsa.kz; consultation@afsa.kz; registrati...,Official public institutional contacts,https://afsa.aifc.kz/aifc-introduces-family-of...,"Use as jurisdiction/regulatory gateway, not a ...",2026-07-01


## Load the other two files

Both use a sheet named `Prospect Longlist` for their main data.

In [3]:
sea_df = pd.read_excel(FILES["sea"], sheet_name="Prospect Longlist")
uzkz_df = pd.read_excel(FILES["uzkz"], sheet_name="Prospect Longlist")

print("SEA:", sea_df.shape)
print("UZKZ:", uzkz_df.shape)

SEA: (14, 30)
UZKZ: (31, 30)


## Compare columns against the master schema

In [4]:
def compare_columns(name, df, base_df=master_df, base_name="master"):
    base_cols = list(base_df.columns)
    other_cols = list(df.columns)

    missing_from_other = [c for c in base_cols if c not in other_cols]  # in master, not in this file
    extra_in_other = [c for c in other_cols if c not in base_cols]      # in this file, not in master
    common = [c for c in base_cols if c in other_cols]

    print(f"=== {base_name} vs {name} ===")
    print(f"common columns ({len(common)}):")
    for c in common:
        print("   ", c)
    print(f"\nin {base_name} but missing from {name} ({len(missing_from_other)}):")
    for c in missing_from_other:
        print("   ", c)
    print(f"\nin {name} but not in {base_name} ({len(extra_in_other)}):")
    for c in extra_in_other:
        print("   ", c)
    print()

    return {"common": common, "missing_from_other": missing_from_other, "extra_in_other": extra_in_other}


sea_comparison = compare_columns("sea", sea_df)
uzkz_comparison = compare_columns("uzkz", uzkz_df)

=== master vs sea ===
common columns (30):
    Region
    Country
    Organisation
    Prospect Category
    HQ / Primary Geography
    Address / Office Location
    Family / Founder / Strategic Nature
    Known Sector Themes
    TBP / Regional Corridor Relevance
    Possible TBP Entry Point
    Recommended Contact Route
    Assigned Lead
    Family Office Fit (20)
    Permanent Capital (20)
    Sector Alignment (20)
    Governance Mindset (15)
    Strategic Adjacency (15)
    Engagement Readiness (10)
    Total Score
    Classification
    Priority
    Pipeline Stage
    Scoring Status
    Public Source URLs
    Notes / Diligence Flags
    Email address
    Contact Email Status
    Contact Email Source URLs
    Contact Email Notes
    Contact Enrichment Date

in master but missing from sea (0):

in sea but not in master (0):

=== master vs uzkz ===
common columns (30):
    Region
    Country
    Organisation
    Prospect Category
    HQ / Primary Geography
    Address / Office Locatio

## Rename corridor columns to match master schema

`Corridor Relevance` (sea) and `Uzbekistan–Kazakhstan Corridor Relevance` (uzkz) are the same field as master's `TBP / Regional Corridor Relevance`, just named per-corridor.

In [5]:
TARGET_CORRIDOR_COL = "TBP / Regional Corridor Relevance"

def rename_corridor_col(df, name):
    if TARGET_CORRIDOR_COL in df.columns:
        print(f"{name}: already has {TARGET_CORRIDOR_COL!r}, nothing to rename")
        return df
    old_col = [c for c in df.columns if "Corridor Relevance" in c][0]
    print(f"{name}: {old_col!r} -> {TARGET_CORRIDOR_COL!r}")
    return df.rename(columns={old_col: TARGET_CORRIDOR_COL})

sea_df = rename_corridor_col(sea_df, "sea")
uzkz_df = rename_corridor_col(uzkz_df, "uzkz")

sea: already has 'TBP / Regional Corridor Relevance', nothing to rename
uzkz: already has 'TBP / Regional Corridor Relevance', nothing to rename


## Add the missing master columns

`Region` and `Assigned Lead` get file-specific fixed values; the rest (`Address / Office Location`, `Pipeline Stage`, `Scoring Status`, `Contact Email Status`, `Contact Email Source URLs`, `Contact Email Notes`, `Contact Enrichment Date`) are master-only tracking fields added blank for now.

In [6]:
MISSING_COLS = [
    "Region",
    "Address / Office Location",
    "Assigned Lead",
    "Pipeline Stage",
    "Scoring Status",
    "Contact Email Status",
    "Contact Email Source URLs",
    "Contact Email Notes",
    "Contact Enrichment Date",
]

for col in MISSING_COLS:
    if col not in sea_df.columns:
        sea_df[col] = pd.NA
    if col not in uzkz_df.columns:
        uzkz_df[col] = pd.NA

# file-specific fixed values
sea_df["Region"] = "Indonesia"
uzkz_df["Region"] = "Central Asia"

sea_df["Assigned Lead"] = "Irene"
uzkz_df["Assigned Lead"] = "Valeriya"

# reorder columns to match master exactly
sea_df = sea_df[master_df.columns]
uzkz_df = uzkz_df[master_df.columns]

print("sea:", sea_df.shape)
print("uzkz:", uzkz_df.shape)

sea: (14, 30)
uzkz: (31, 30)


## Re-compare columns (should now be a full match)

In [7]:
sea_comparison = compare_columns("sea", sea_df)
uzkz_comparison = compare_columns("uzkz", uzkz_df)

=== master vs sea ===
common columns (30):
    Region
    Country
    Organisation
    Prospect Category
    HQ / Primary Geography
    Address / Office Location
    Family / Founder / Strategic Nature
    Known Sector Themes
    TBP / Regional Corridor Relevance
    Possible TBP Entry Point
    Recommended Contact Route
    Assigned Lead
    Family Office Fit (20)
    Permanent Capital (20)
    Sector Alignment (20)
    Governance Mindset (15)
    Strategic Adjacency (15)
    Engagement Readiness (10)
    Total Score
    Classification
    Priority
    Pipeline Stage
    Scoring Status
    Public Source URLs
    Notes / Diligence Flags
    Email address
    Contact Email Status
    Contact Email Source URLs
    Contact Email Notes
    Contact Enrichment Date

in master but missing from sea (0):

in sea but not in master (0):

=== master vs uzkz ===
common columns (30):
    Region
    Country
    Organisation
    Prospect Category
    HQ / Primary Geography
    Address / Office Locatio

## Write changes back to the source files

Back up each file once, then replace only the `Prospect Longlist` sheet in place (other sheets like `Shortlisted` / `Scoring Model` are preserved untouched).

In [8]:
import shutil

for key in ["sea", "uzkz"]:
    backup_path = FILES[key].with_name(FILES[key].stem + ".backup.xlsx")
    if not backup_path.exists():
        shutil.copy(FILES[key], backup_path)
        print("Backed up ->", backup_path)
    else:
        print("Backup already exists, skipping ->", backup_path)

Backup already exists, skipping -> ..\TBP_SEA_Medium sized.backup.xlsx
Backup already exists, skipping -> ..\TBP_UZ-KZ_Independent_Corridor_Investor_Longlist.backup.xlsx


In [9]:
with pd.ExcelWriter(FILES["sea"], engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    sea_df.to_excel(writer, sheet_name="Prospect Longlist", index=False)

with pd.ExcelWriter(FILES["uzkz"], engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    uzkz_df.to_excel(writer, sheet_name="Prospect Longlist", index=False)

print("Done writing updated 'Prospect Longlist' sheets.")

Done writing updated 'Prospect Longlist' sheets.


## Verify: re-read from disk

In [10]:
import openpyxl

for key in ["sea", "uzkz"]:
    wb = openpyxl.load_workbook(FILES[key], read_only=True)
    print(key, "sheets:", wb.sheetnames)

sea_reloaded = pd.read_excel(FILES["sea"], sheet_name="Prospect Longlist")
uzkz_reloaded = pd.read_excel(FILES["uzkz"], sheet_name="Prospect Longlist")

compare_columns("sea (reloaded)", sea_reloaded)
compare_columns("uzkz (reloaded)", uzkz_reloaded)

sea sheets: ['Prospect Longlist', 'Shortlisted']
uzkz sheets: ['Executive Summary', 'Prospect Longlist', 'Scored Shortlist', 'Scoring Model']


=== master vs sea (reloaded) ===
common columns (30):
    Region
    Country
    Organisation
    Prospect Category
    HQ / Primary Geography
    Address / Office Location
    Family / Founder / Strategic Nature
    Known Sector Themes
    TBP / Regional Corridor Relevance
    Possible TBP Entry Point
    Recommended Contact Route
    Assigned Lead
    Family Office Fit (20)
    Permanent Capital (20)
    Sector Alignment (20)
    Governance Mindset (15)
    Strategic Adjacency (15)
    Engagement Readiness (10)
    Total Score
    Classification
    Priority
    Pipeline Stage
    Scoring Status
    Public Source URLs
    Notes / Diligence Flags
    Email address
    Contact Email Status
    Contact Email Source URLs
    Contact Email Notes
    Contact Enrichment Date

in master but missing from sea (reloaded) (0):

in sea (reloaded) but not in master (0):

=== master vs uzkz (reloaded) ===
common columns (30):
    Region
    Country
    Organisation
    Prospect Category
    HQ / Pr

{'common': ['Region',
  'Country',
  'Organisation',
  'Prospect Category',
  'HQ / Primary Geography',
  'Address / Office Location',
  'Family / Founder / Strategic Nature',
  'Known Sector Themes',
  'TBP / Regional Corridor Relevance',
  'Possible TBP Entry Point',
  'Recommended Contact Route',
  'Assigned Lead',
  'Family Office Fit (20)',
  'Permanent Capital (20)',
  'Sector Alignment (20)',
  'Governance Mindset (15)',
  'Strategic Adjacency (15)',
  'Engagement Readiness (10)',
  'Total Score',
  'Classification',
  'Priority',
  'Pipeline Stage',
  'Scoring Status',
  'Public Source URLs',
  'Notes / Diligence Flags',
  'Email address',
  'Contact Email Status',
  'Contact Email Source URLs',
  'Contact Email Notes',
  'Contact Enrichment Date'],
 'missing_from_other': [],
 'extra_in_other': []}

## Merge into one master dataframe and write Master_List.csv

In [11]:
master_df["Source File"] = FILES["master"].name
sea_df["Source File"] = FILES["sea"].name
uzkz_df["Source File"] = FILES["uzkz"].name

merged_df = pd.concat([master_df, sea_df, uzkz_df], ignore_index=True)

print(merged_df.shape)
print(merged_df["Source File"].value_counts())
merged_df.head()

(158, 31)
Source File
TBP_Family_Office_Generated_Lists_Fresh_Master.xlsx      113
TBP_UZ-KZ_Independent_Corridor_Investor_Longlist.xlsx     31
TBP_SEA_Medium sized.xlsx                                 14
Name: count, dtype: int64


,Region,Country,Organisation,Prospect Category,HQ / Primary Geography,Address / Office Location,Family / Founder / Strategic Nature,Known Sector Themes,TBP / Regional Corridor Relevance,Possible TBP Entry Point,...,Pipeline Stage,Scoring Status,Public Source URLs,Notes / Diligence Flags,Email address,Contact Email Status,Contact Email Source URLs,Contact Email Notes,Contact Enrichment Date,Source File
0,Central Asia,Kazakhstan,Verny Capital / Bulat Utemuratov,Private investment group / family-capital styl...,Almaty / Astana,NaN,Private investment group / family-capital styl...,Family office services / private wealth / stew...,Leader in private investments in Kazakhstan; i...,Central Asia / Middle Corridor; Uzbekistan-Kaz...,...,Identified,Indicative scored – review required,https://vernycapital.com/en/bulat-utemuratov-i...,Apply enhanced due diligence where political e...,info@vernycapital.com,Official/public email - verify before outreach,https://vernycapital.com/en/bulat-utemuratov-i...,Use general office email or warm introduction;...,2026-07-01,TBP_Family_Office_Generated_Lists_Fresh_Master...
1,Central Asia,Kazakhstan,BI Group / Aidyn Rakhimbayev,"Investment, development, construction and educ...",Astana,NaN,"Investment, development, construction and educ...",Family office services / private wealth / stew...,"City development, real estate, construction an...",Central Asia / Middle Corridor; Uzbekistan-Kaz...,...,Identified,Indicative scored – review required,https://bi.group/en/company,Apply enhanced due diligence where political e...,ipm@bi.group,Official/public email - verify before outreach,https://bi.group/en/company,Use institutional or partnership route; verify...,2026-07-01,TBP_Family_Office_Generated_Lists_Fresh_Master...
2,Central Asia,Kazakhstan,Lancaster Group,Diversified international holding company,Almaty,NaN,Diversified international holding company,"Mining, infrastructure, energy, financial serv...","Industrial infrastructure, oil and gas service...",Central Asia / Middle Corridor; Uzbekistan-Kaz...,...,Identified,Indicative scored – review required,https://lancasterholding.com/en,Apply enhanced due diligence where political e...,info@lgk.kz,Official/public email - verify before outreach,https://lancasterholding.com/en,Use general office email; verify current conta...,2026-07-01,TBP_Family_Office_Generated_Lists_Fresh_Master...
3,Central Asia,Kazakhstan,Resmi Group,Diversified investment holding,Kazakhstan / Central Asia,NaN,Diversified investment holding,Family office services / private wealth / stew...,Diversified investment holding operating in Ka...,Central Asia / Middle Corridor; Uzbekistan-Kaz...,...,Identified,Indicative scored – review required,https://kz.linkedin.com/company/group-of-compa...,Apply enhanced due diligence where political e...,inform@resmi.kz,Official/public email - verify before outreach,https://kase.kz/en/listing/issuers/RESC/,Issuer/contact source; verify best route for c...,2026-07-01,TBP_Family_Office_Generated_Lists_Fresh_Master...
4,Central Asia,Kazakhstan,AIFC Family Office Framework,Family office jurisdiction / structuring platform,Astana,NaN,Family office jurisdiction / structuring platform,Family office services / private wealth / stew...,Institutional route for structuring Central As...,Central Asia / Middle Corridor; Uzbekistan-Kaz...,...,Identified,Indicative scored – review required,https://afsa.aifc.kz/aifc-introduces-family-of...,Apply enhanced due diligence where political e...,info@afsa.kz; consultation@afsa.kz; registrati...,Official public institutional contacts,https://afsa.aifc.kz/aifc-introduces-family-of...,"Use as jurisdiction/regulatory gateway, not a ...",2026-07-01,TBP_Family_Office_Generated_Lists_Fresh_Master...


In [12]:
OUTPUT_PATH = Path("data") / "Master_List.csv"
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

merged_df.to_csv(OUTPUT_PATH, index=False)
print("Wrote", OUTPUT_PATH.resolve(), "rows:", len(merged_df))

Wrote C:\Users\USER\Desktop\TBP\tbp-dashboard\data\Master_List.csv rows: 158
